## 1. Overview of k-Nearest Neighbors (k-NN)

k-NN is a foundational machine learning algorithm that carries a profound philosophy about data space. Its foundation is the **Smoothing Assumption:** data points with similar features will tend to converge closely in a multi-dimensional space.

What makes k-NN different from Linear Regression or Neural Networks lies in two characteristics:
- **Lazy Learning:** The model does not try to solve a general mathematical equation during training.
- **Instance-based learning:** The model simply memorizes the entire dataset into memory. All complex calculations only erupt during the prediction phase (inference), when it must measure the distance from a new data point to all existing points to find the $k$ nearest neighbors.

## 2. How to k-NN model work?

To make the algorithm operate accurately, we must solve two fundamental problems: synchronizing the scale and defining proximity.

### A. Synchronizing scale with Data Standardization (Scaling)

The essence of k-NN is measuring geometric distance. If the features do not share the same scale, the mathematical space will be completely distorted.

Example illustrating the collapse without standardization:

Suppose we classify customers based on Age and Salary.
- Customer A: 20 years old, salary 15,000,000 VND.
- Customer B: 60 years old, salary 15,050,000 VND.
- New Customer Q: 20 years old, salary 15,050,000 VND.

Logically, Q is almost a clone of A because a difference of 50,000 VND is meaningless, while Q is 40 years apart from B. However, applying the original Euclidean distance:
- Distance from Q to A: $\sqrt{0^2 + 50000^2} = 50000$
- Distance from Q to B: $\sqrt{(-40)^2 + 0^2} = 40$

The algorithm concludes Q is right next to B and far from A. The reason is that the Salary variable has an enormous amplitude; when squared, it creates a massive number that completely swallows the classification meaning of the Age variable.

**How to fix it:** Use **Min-Max Scaling** to compress both variables into the range $[0, 1]$. Now, the 40-year difference will hold a large value (like 0.95), while the 50,000 VND difference will be almost 0 (like 0.001). The geometric distance will reflect the true nature, classifying Q correctly into A's group.

### B. Distance Metrics

The most fundamental form to measure disparity is the Minkowski distance:
$$
D(x, y) = \left(\sum_{i=1}^{n} |x_{i} - y_{i}|^{p} \right)^{\frac{1}{p}}
$$
Depending on the parameter $p$, we have different application strategies:
- **Euclidean Distance ($p=2$):** Measures the straight line. Because of the squaring operation, it heavily penalizes large differences. Suitable for continuous, evenly scaled data with little noise.
- **Manhattan Distance ($p=1$):** Measures the zigzag path along coordinate axes. By using absolute values, it resists noise better and maintains good separation in high-dimensional spaces.
- **Cosine Similarity:** Measures the angle between two vectors instead of physical distance. This is the optimal choice for recommendation systems or text mining, where the direction of the data is more important than the absolute magnitude.

### C. Inference Mechanism

Due to its Lazy Learning nature, the entire computational load of k-NN occurs during the inference phase. When a new data point (query $Q$) is introduced, the processing pipeline is executed through three core steps:

**Step 1: Distance Measurement**

The model calculates the distance from point $Q$ to all data points in the training set (using Euclidean, Manhattan, or Cosine metrics on the standardized space). In the Brute Force approach, the algorithm must perform calculations against every single point present in the dataset.

**Step 2: Neighbor Extraction**

The newly calculated distances are sorted in ascending order. The algorithm then selects exactly $k$ data points with the smallest distances. This forms the foundational neighbor set used to infer the result for $Q$.

**Step 3: Decision Rule**

How k-NN dictates the final outcome depends on the specific type of problem being solved:

- **Classification:** Applies the Majority Voting principle. The label appearing most frequently within the $k$ neighbors is assigned to $Q$.
- **Regression:** Predicts a continuous value by calculating the Mean or Median of the target values from the $k$ neighbors.

## 3. Optimizing the hyperparameter k

Choosing $k$ is a battle to find the balance point between Bias and Variance. A small $k$ causes Overfitting (sensitive to noise), while a large $k$ causes Underfitting (flattening features).

To find the perfect $k$, we combine two techniques:
- **K-Fold Cross-Validation:** Divides the data into multiple parts, alternately using one part for validation and the rest for training. This evaluates generalization capacity most honestly, preventing the model from deceiving itself through rote memorization.
- **Elbow Method:** Plots the error rate against each value of $k$. The elbow point is where the error stops dropping steeply and starts leveling off. That is the optimal $k$ value.

## 4. Improvements with Distance-Weighted k-NN

The basic k-NN model uses Majority Voting, where every neighbor has an equal vote. This creates risks when data density is imbalanced.

**The flaw of traditional k-NN:**

Need to classify point Q with $k=5$. There are 3 points of Class A located quite far away (distances 10, 12, 15) and 2 points of Class B located right next to it (distances 1, 2). The traditional algorithm gives Class A the win with 3 votes. This decision is absurd because Class A is merely relying on its crowded surrounding population to overwhelm point Q.

**The Distance-Weighted k-NN solution:**

Assign voting weights inversely proportional to distance ($w = 1/d$).
- Power of Class A: $1/10 + 1/12 + 1/15 \approx 0.25$
- Power of Class B: $1/1 + 1/2 = 1.5$

Class B completely crushes Class A. The model smooths out the decision boundary and neutralizes the injustice caused by data density.

## 5. Evaluating Pros and Cons

**Pros:**
- **Non-parametric:** Does not require the data to follow any mathematical distribution.
- **Intuitive:** The algorithm is easy to understand, easy to implement, and results are highly interpretable.
- **Continuous learning:** Updating new data does not require training from scratch.
- **Naturally multi-class:** Handles problems with more than two classification labels smoothly.

**Cons:**
- **Real-time latency:** Prediction cost is extremely expensive ($O(N \times D)$).
- **Hardware consumption:** Requires large RAM capacity to store the entire training dataset.
- **Sensitive:** Easily deceived by noisy data or scale disparities.
- **Curse of Dimensionality:** Performance collapses when the number of features increases too high.

## 6. Overcoming Limitations with Spatial Structures

To escape the slow sequential scanning loop, we must use smarter storage structures.
- **KD-Tree (K-Dimensional Tree):** Divides space using perpendicular hyperplanes. Uses backtracking to prune redundant search branches, reducing time to a logarithmic level. The weakness is that it easily paralyzes when data exceeds 20 dimensions.
- **Ball Tree:** Wraps data in hyper-spheres. Applies the triangle inequality to skip an entire sphere if its center is too far from the query point. More flexible than KD-Tree in medium-dimensional spaces.
- **ANN (Approximate Nearest Neighbors):** The weapon of practical industrial AI. Trades a small percentage of accuracy to increase speed tens of thousands of times. Uses binary hashing (LSH), builds highway networks (HNSW), and compresses vectors (Product Quantization) to smoothly handle billions of data points.

## Code k-Nearest Neighbors (k-NN) from scratch

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn import datasets
from sklearn.neighbors import KNeighborsClassifier

In [2]:
class KNN:
    def __init__(self, k=3, p=2):
        self.k = k
        self.p = p

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def minkowski_distance(self, x1, x2):
        return np.sum(np.abs(x1 - x2) ** self.p) ** (1 / self.p)
    
    def _predict(self, x):
        # compute the distances
        # distances = [self.minkowski_distance(x, x_train) for x_train in self.X_train]
        distances = np.sum(np.abs(self.X_train - x) ** self.p, axis=1) ** (1 / self.p)
        # get k nearest neighbors
        k_indices = np.argpartition(distances, self.k)[:self.k]
        k_nearest_labels = [self.y_train[i] for i in k_indices]

        # majority vote
        most_common = max(set(k_nearest_labels), key=k_nearest_labels.count)
        return most_common

    def predict(self, X):
        return [self._predict(x) for x in X]


In [3]:
iris = datasets.load_iris()

X, y = iris.data, iris.target

In [4]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (150, 4)
y shape: (150,)


In [5]:
X[:5]

array([[5.1, 3.5, 1.4, 0.2],
       [4.9, 3. , 1.4, 0.2],
       [4.7, 3.2, 1.3, 0.2],
       [4.6, 3.1, 1.5, 0.2],
       [5. , 3.6, 1.4, 0.2]])

In [6]:
y[:5]

array([0, 0, 0, 0, 0])

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=50, random_state=42)

In [8]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
knn = KNN(k=3)
knn.fit(X_train_scaled, y_train)

In [10]:
y_pred = knn.predict(X_test_scaled)

In [11]:
acc_scrore = accuracy_score(y_test, y_pred)
print("Accuracy:", acc_scrore)

Accuracy: 0.98


## Code using Sklearn

In [12]:
knn = KNeighborsClassifier(n_neighbors=3, p=2)
knn.fit(X_train_scaled, y_train)

,n_neighbors,3
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [13]:
y_pred = knn.predict(X_test_scaled)

In [14]:
acc_scrore = accuracy_score(y_test, y_pred)
print("Accuracy:", acc_scrore)

Accuracy: 0.98


In [15]:
knn = KNeighborsClassifier(n_neighbors=3, p=2, weights="distance")
knn.fit(X_train_scaled, y_train)

,n_neighbors,3
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [16]:
y_pred = knn.predict(X_test_scaled)

In [17]:
acc_scrore = accuracy_score(y_test, y_pred)
print("Accuracy:", acc_scrore)

Accuracy: 0.98


## Digit Recognition with MNIST dataset

In [18]:
X = np.load("MNIST/X.npy")
y = np.load("MNIST/y.npy")

In [19]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (5000, 400)
y shape: (5000, 1)


In [20]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_values = range(1, 20, 2)
cv_scores = []

print("K-Fold Cross Validation:")
for k in k_values:
    fold_accuracies = []

    for train_index, val_index in kf.split(X_train):
        X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        scaler = MinMaxScaler()
        X_train_fold_scaled = scaler.fit_transform(X_train_fold)
        X_val_fold_scaled = scaler.transform(X_val_fold)

        knn = KNeighborsClassifier(n_neighbors=k, p=2, weights="distance")

        knn.fit(X_train_fold_scaled, y_train_fold.ravel())

        y_pred = knn.predict(X_val_fold_scaled)

        acc_score = accuracy_score(y_val_fold, y_pred)
        fold_accuracies.append(acc_score)
    print(f"k={k}: Fold Accuracies: {fold_accuracies}, Mean CV Accuracy: {np.mean(fold_accuracies):.4f}")

    cv_scores.append(np.mean(fold_accuracies))

K-Fold Cross Validation:
k=1: Fold Accuracies: [0.93, 0.935, 0.9525, 0.935, 0.935], Mean CV Accuracy: 0.9375
k=3: Fold Accuracies: [0.93, 0.94, 0.9575, 0.93625, 0.94375], Mean CV Accuracy: 0.9415
k=5: Fold Accuracies: [0.92875, 0.9425, 0.94625, 0.9425, 0.94], Mean CV Accuracy: 0.9400
k=7: Fold Accuracies: [0.93125, 0.93375, 0.9375, 0.9425, 0.93875], Mean CV Accuracy: 0.9367
k=9: Fold Accuracies: [0.9175, 0.9275, 0.93125, 0.93875, 0.93875], Mean CV Accuracy: 0.9307
k=11: Fold Accuracies: [0.9125, 0.93, 0.92875, 0.93625, 0.93375], Mean CV Accuracy: 0.9283
k=13: Fold Accuracies: [0.9025, 0.9175, 0.92875, 0.935, 0.9325], Mean CV Accuracy: 0.9233
k=15: Fold Accuracies: [0.90125, 0.9125, 0.92625, 0.93375, 0.93625], Mean CV Accuracy: 0.9220
k=17: Fold Accuracies: [0.89625, 0.90875, 0.91875, 0.93125, 0.9325], Mean CV Accuracy: 0.9175
k=19: Fold Accuracies: [0.89625, 0.905, 0.91125, 0.9225, 0.93125], Mean CV Accuracy: 0.9133


In [ ]:
best_k = k_values[np.argmax(cv_scores)]
print(f"Best k: {best_k} with CV Accuracy: {max(cv_scores):.4f}")

Best k: 3 with CV Accuracy: 0.9415


In [23]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [24]:
knn = KNeighborsClassifier(n_neighbors=best_k, p=2, weights="distance")
knn.fit(X_train_scaled, y_train.ravel())

,n_neighbors,3
,weights,'distance'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [25]:
y_pred = knn.predict(X_test_scaled)

In [26]:
acc_score = accuracy_score(y_test, y_pred)
print(f"Test Accuracy with k={best_k}: {acc_score:.4f}")

Test Accuracy with k=3: 0.9470
